In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from power_systems_ops.read import login, read_tables, navigate

In [3]:
email = "coned@colorado.edu"
password = "2025NYTeam"
driver = login(email, password)

In [4]:
tables = read_tables(driver)

In [8]:
from selenium.webdriver.common.by import By

In [5]:
for k, table in tables.items():
    print(k)
    print(table)


dispatch
                Type Capacity (MW) Reliability (%) Efficiency (%)  \
Name                                                                
e1           nuclear        800.00            93.7          33.53   
e2        powderCoal        700.00            95.8          40.19   
e4              wind        150.00            95.6            N/A   
e5    naturalgasCCGT        300.00            92.0          51.06   
e7    naturalgasOCGT         50.00            92.9          36.08   
e9             solar         20.00            93.6           0.00   
e10          biomass         90.00             N/A          40.59   
e3              wind         50.00            95.8            N/A   
e6    naturalgasCCGT        600.00            94.5          53.99   
e8    naturalgasOCGT         50.00             N/A          37.40   

     Loan payment (M€/year) Remaining payments (years)  \
Name                                                     
e1                    308.1                   

<selenium.webdriver.remote.webelement.WebElement (session="889ab54255da9cde20e83938c2134d07", element="f.864CB1F3D29F371EA75EDA22B02E4028.d.D691671E41FF8AECEC1B29EA742E6F9C.e.27")>

In [13]:
pp_link = driver.find_element(By.XPATH, "//a[@title='Power plants']")
pp_link.find_element(By.TAG_NAME, 'img').click()

In [22]:
for link in links:
    print(link.get_attribute('href'))
    print(link.get_attribute('title'))

https://emsg2.tbm.tudelft.nl/cu-boulder-2025/instance_select.jsp
Other instances
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/logout.jsp
Logout
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/dashboard.jsp
Dashboard
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/stocks_and_trends.jsp
Stocks & trends
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/plantsOverview.jsp
Power plants
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/plantsAuctioning.jsp
Build, decommision or trade plant
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/bank.jsp
Bank account
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/electricity_market.jsp
Electricity market
https://emsg2.tbm.tudelft.nl/cu-boulder-2025/player/balancing_market.jsp
Balancing market

Delft University of Technology and Fields of View


In [20]:
import pandas as pd

In [56]:
TABLE_NAME_TO_KEY = {
    "Power plant portfolio": "dispatch",
    "Electricity generation and fuel consumption in the past round": "last_generation",
    "Electricity production per plant": "production",
    "Power plant availability": "availability",
    "Overview of power plants": "overview"
}

INDEX_NAMES = {
    "dispatch": "Name",
    # "last_generation": "Plant",
    # "availability": "Plant",
    # "overview": "Plant name"
}
    
    
    
    

In [60]:
driver.find_element(By.TAG_NAME, "table").find_elements(By.TAG_NAME, 'th')

[<selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2daa6657b9685c", element="f.8835B30BC8830730D0ED6E8BD6080507.d.DB579C6F2A0BCCBC3DB128ED0CAEBE64.e.267")>,
 <selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2daa6657b9685c", element="f.8835B30BC8830730D0ED6E8BD6080507.d.DB579C6F2A0BCCBC3DB128ED0CAEBE64.e.269")>,
 <selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2daa6657b9685c", element="f.8835B30BC8830730D0ED6E8BD6080507.d.DB579C6F2A0BCCBC3DB128ED0CAEBE64.e.271")>,
 <selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2daa6657b9685c", element="f.8835B30BC8830730D0ED6E8BD6080507.d.DB579C6F2A0BCCBC3DB128ED0CAEBE64.e.188")>,
 <selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2daa6657b9685c", element="f.8835B30BC8830730D0ED6E8BD6080507.d.DB579C6F2A0BCCBC3DB128ED0CAEBE64.e.63")>,
 <selenium.webdriver.remote.webelement.WebElement (session="f9623eddaa8dbefd2f2da

In [84]:
def filter_empty(strings):
    return [s for s in strings if s]
    
def join_2_rows(header_elem, key):
    rows = header_elem.find_elements(By.TAG_NAME, "tr")
    assert len(rows) == 2
    row1 = [h.text for h in rows[0].find_elements(By.TAG_NAME, 'th')]
    row2 = [h.text for h in rows[1].find_elements(By.TAG_NAME, 'th')]
    if key == "last_generation":
        # Extend the final column of the first row, since it pertains to each
        # of the last three columns of row2...
        
        row1 += [row1[-1], row1[-1]]
    return [' -- '.join(filter_empty([h1, h2])) for h1, h2 in zip(row1, row2)]
    
def extract_column_names(table, key):

    header_elem = table.find_element(By.TAG_NAME, "thead")
    # Add special logic for each table...
    if key in ['last_generation', 'availability']:
        return join_2_rows(header_elem, key)

    column_names = [header.text for header in header_elem.find_elements(By.TAG_NAME, "th")]

    if key == "overview":
        column_names.append("status")
    
    return column_names

def extract_rows(table, key):
    body = table.find_element(By.TAG_NAME, "tbody")
    rows = body.find_elements(By.TAG_NAME, "tr")
    if key == "overview":
        return extract_rows_overview(rows)
        
    data = []
    for row in rows:
        cells = row.find_elements(By.TAG_NAME, "td")
        row_data = [cell.text for cell in cells]
        data.append(row_data)
    return data


HEADER_TO_KEY = {
    "Operational power plants": "operational",
    "Power plants under construction": "under construction",
    "Dismantled power plants": "dismantled"
}

def extract_rows_overview(rows):
    data = []
    current_status = None
    for row in rows:
        headers = row.find_elements(By.TAG_NAME, "th")
        if headers:
            assert len(headers) == 1
            current_status = HEADER_TO_KEY[headers[0].text]
            continue
        row_data = [cell.text for cell in row.find_elements(By.TAG_NAME, "td")]
        if current_status == "under construction":
            # For some reason data in the section is missing a cell for "round dismantled".
            row_data.append('')
        row_data.append(current_status)
        data.append(row_data)
    return data
        
    
def extract_table(table, key):
    
    # Extract table headers and rows
    headers = extract_column_names(table, key)
    
    # Prepare data for DataFrame (skip the header row)
    rows = extract_rows(table, key)
    
    # Create a DataFrame
    df = pd.DataFrame(rows, columns=headers)
    if index_name is not None:
        df = df.set_index(index_name)
    return df

In [85]:
tables = {}

table_containers = driver.find_elements(By.XPATH, "//div[@class='component table left white']")

for div in table_containers:
    name = div.find_element(By.TAG_NAME, 'h4').text
    key = TABLE_NAME_TO_KEY[name]
    print(key)
    tables_found = div.find_elements(By.TAG_NAME, "table")
    if len(tables_found) > 1:
        raise ValueError("Found multiple tables within single div!!!")
    elif len(tables_found) == 1:
        index_name = INDEX_NAMES.get(key)
        df = extract_table(tables_found[0], key)
        print(df)
        tables[key] = df

dispatch
                Type Capacity (MW) Reliability (%) Efficiency (%)  \
Name                                                                
e1           nuclear        800.00            93.7          33.53   
e2        powderCoal        700.00            95.8          40.19   
e4              wind        150.00            95.6            N/A   
e5    naturalgasCCGT        300.00            92.0          51.06   
e7    naturalgasOCGT         50.00            92.9          36.08   
e9             solar         20.00            93.6           0.00   
e10          biomass         90.00             N/A          40.59   
e3              wind         50.00            95.8            N/A   
e6    naturalgasCCGT        600.00            94.5          53.99   
e8    naturalgasOCGT         50.00             N/A          37.40   

     Loan payment (M€/year) Remaining payments (years)  \
Name                                                     
e1                    308.1                   

In [52]:
tables['last_generation']

,Name,Type,Capacity (MW),Reliability (%),Efficiency (%),Loan payment (M€/year),Remaining payments (years),Fixed O&M costs (M€/year),Status,First round active,Priority
0,e1,nuclear,800.00,93.7,33.53,308.1,19,91.8,available,-9,1
1,e2,powderCoal,700.00,95.8,40.19,65.5,19,85.6,available,0,2
2,e4,wind,150.00,95.6,N/A,30.7,14,7.1,available,-1,3
3,e5,naturalgasCCGT,300.00,92.0,51.06,20.6,14,20.3,available,-13,4
4,e7,naturalgasOCGT,50.00,92.9,36.08,3.4,9,1.3,available,-11,5
5,e9,solar,20.00,93.6,0.00,1.4,14,0.4,available,-3,6
6,e10,biomass,90.00,N/A,40.59,8.8,20,N/A,under construction,2,7
7,e3,wind,50.00,95.8,N/A,9.9,14,2.4,available,0,8
8,e6,naturalgasCCGT,600.00,94.5,53.99,38.5,14,38.7,available,-6,9
9,e8,naturalgasOCGT,50.00,N/A,37.40,3.2,9,1.3,in maintenance,-5,10


In [ ]:
table = driver.find_element(By.XPATH, f"//table[@id='{id_}']")

In [30]:
dispatch_stack = extract_table("powerplants", "Name")

In [32]:
dispatch_stack

,Type,Capacity (MW),Reliability (%),Efficiency (%),Loan payment (M€/year),Remaining payments (years),Fixed O&M costs (M€/year),Status,First round active,Priority
Name,,,,,,,,,,
e1,nuclear,800.00,93.7,33.53,308.1,19,91.8,available,-9,1
e2,powderCoal,700.00,95.8,40.19,65.5,19,85.6,available,0,2
e4,wind,150.00,95.6,N/A,30.7,14,7.1,available,-1,3
e5,naturalgasCCGT,300.00,92.0,51.06,20.6,14,20.3,available,-13,4
e7,naturalgasOCGT,50.00,92.9,36.08,3.4,9,1.3,available,-11,5
e9,solar,20.00,93.6,0.00,1.4,14,0.4,available,-3,6
e10,biomass,90.00,N/A,40.59,8.8,20,N/A,under construction,2,7
e3,wind,50.00,95.8,N/A,9.9,14,2.4,available,0,8
e6,naturalgasCCGT,600.00,94.5,53.99,38.5,14,38.7,available,-6,9


In [15]:
link

''

In [14]:
dir(link)

['__abstractmethods__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_execute',
 '_id',
 '_parent',
 '_upload',
 'accessible_name',
 'aria_role',
 'clear',
 'click',
 'find_element',
 'find_elements',
 'get_attribute',
 'get_dom_attribute',
 'get_property',
 'id',
 'is_displayed',
 'is_enabled',
 'is_selected',
 'location',
 'location_once_scrolled_into_view',
 'parent',
 'rect',
 'screenshot',
 'screenshot_as_base64',
 'screenshot_as_png',
 'send_keys',
 'shadow_root',
 'size',
 'submit',
 'tag_name',
 'text',
 'value_of_css_property']

In [7]:
help(webdriver.Chrome)

Help on class WebDriver in module selenium.webdriver.chrome.webdriver:

class WebDriver(selenium.webdriver.chromium.webdriver.ChromiumDriver)
 |  WebDriver(options: selenium.webdriver.chrome.options.Options = None, service: selenium.webdriver.chrome.service.Service = None, keep_alive: bool = True) -> None
 |
 |  Controls the ChromeDriver and allows you to drive the browser.
 |
 |  Method resolution order:
 |      WebDriver
 |      selenium.webdriver.chromium.webdriver.ChromiumDriver
 |      selenium.webdriver.remote.webdriver.WebDriver
 |      selenium.webdriver.remote.webdriver.BaseWebDriver
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(self, options: selenium.webdriver.chrome.options.Options = None, service: selenium.webdriver.chrome.service.Service = None, keep_alive: bool = True) -> None
 |      Creates a new instance of the chrome driver. Starts the service and
 |      then creates new instance of chrome driver.
 |
 |      :Args:
 |       - options - this ta

In [6]:
driver.quit()